In [1]:
import pandas as pd
import numpy as np
import great_expectations as gx
from pathlib import Path


In [3]:
# read in .csv file

project_root = Path.cwd().parent
#("Telco-Customer-Churn.csv")
data_dir = project_root /"src" / "data" / "raw"
f_path = data_dir / "Telco-Customer-Churn.csv"
print(project_root)

# path = Path(r'C:\Users\steph\Documents\GitHub\Smokey_pipeline_mazes')
# f_path = path.joinpath('src', 'data', 'raw', 'Telco-Customer-Churn.csv')
# print(f_path)

with open(f_path, 'r') as f:
    df = pd.read_csv(f)
    print(df.head())

c:\Users\steph\Documents\GitHub\ML_pipe
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Streamin

In [ ]:
# perform quick data validation checks
# setup GX context:

context = gx.get_context()

# Define data source/how to connect to data
data_source_name = 'pandas'
try:
    data_source = context.data_sources.add_pandas(name = data_source_name)
except gx.exceptions.DataContextError:
    data_source = context.data_sources.get('pandas')
#data_source = context.data_sources.add_pandas(name = 'pandas')


# Add data asset

asset_name = 'Telco-Customer-Churn'
try:
    data_asset = data_source.get_asset(asset_name)
except gx.exceptions.DataContextError:
    data_asset = data_source.add_dataframe_asset(name = asset_name)

# Create a batch definition (entire df)
batch_def_name = 'Telco-batch'
try:
    batch_definition = data_asset.get_batch_definition(batch_def_name)
except gx.exceptions.DataContextError:
    batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_def_name)

batch = batch_definition.get_batch(batch_parameters= {"dataframe": df})



In [4]:
print(data_source)

assets:
  - batch_definitions:
      - id: 6cca0c33-36a6-4f37-9c44-f54073821364
        name: Telco-batch
        partitioner: null
    batch_metadata: {}
    id: 2266c1c6-8c1f-49ae-9022-699633fb915a
    name: Telco-Customer-Churn
    type: dataframe
id: f3a997c6-0fa0-4503-96c1-d1f928e1189b
name: pandas
type: pandas



In [ ]:
from great_expectations.expectations import (
    ExpectColumnValuesToBeUnique,
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
)


# create expectations suite:

suite_name = "telco_expectations"
suite = gx.ExpectationSuite(name = suite_name)

#suite = context.suites.add(suite)

# Primary Key validations:
suite.add_expectation(ExpectColumnValuesToNotBeNull(column = "customerID"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column = "churn"))
suite.add_expectation(ExpectColumnValuesToBeUnique(column = "customerID"))


ExpectColumnValuesToBeUnique(id='74b59193-7fd5-471f-863d-9bff3d69484f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customerID', mostly=1, row_condition=None, condition_parser=None)

In [6]:
# Add numeric Validations:

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

#coerce TotalCharges to numeric, forcing errors to NaN:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')

for col in numeric_cols:
    min_val = float(df[col].dropna().min())
    max_val = float(df[col].dropna().max())

    suite.add_expectation(
        ExpectColumnValuesToBeBetween(
            column = col,
            min_value = min_val,
            max_value = max_val
        )
    )

In [7]:
# Add value-set validations for categorical columns:
value_set_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines", 
    "InternetService", "OnlineSecurity", "OnlineBackup", "SeniorCitizen", 
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", 
    "Contract", "PaperlessBilling", "PaymentMethod", "Churn"
]

for col in value_set_cols:
    unique_vals = df[col].dropna().unique().tolist()

    suite.add_expectation(
        ExpectColumnValuesToBeInSet(
            column = col,
            value_set = unique_vals
        )
    )


# Finally, save the suite to the data context:

context.suites.add_or_update(suite)

print(f"Suite '{suite_name}' successfully created with {len(suite.expectations)} expectations.")

Suite 'telco_expectations' successfully created with 23 expectations.


### Create Validation Definition

In [ ]:
# # retrieve expectation suite from context ????????????????????????

# existing_suite_name = (
#     "telco_expectations"
# )
# suite = context.suites.get(name = existing_suite_name)

In [ ]:
# # retrieve Batch Definition that describes the data: >????????????????????????????????

# data_source_name = "pandas"
# data_asset_name = "Telco-Customer-Churn"
# batch_definition_name = "Telco-batch"

# batch_definition = (
#     context.data_sources.get(data_source_name)
#     .get_asset(data_asset_name)
#     .get_batch_definition(batch_definition_name)
# )


In [8]:
# Create a Validation Definition (bridges specific batch of data to an Expectation Suite):
definition_name = "telco_validation"

validation_definition = gx.ValidationDefinition(
    data = batch_definition, suite = suite, name = definition_name
)


In [10]:
# Save this validation definition to the data context:
#validation_definition = context.validation_definitions.add(validation_definition)

In [ ]:
# # Create Action that the Checkpoint will perform after running the validation definition:
# from great_expectations.checkpoint.actions import EmailNotificationAction

# action_list = [
#     EmailNotificationAction(
#         name = "send_email_notification_on_failed_expectations",


# ]

In [ ]:
# Create a Checkpoint to run the validation definition:

try:
    checkpoint = context.checkpoints.get("telco_checkpoint")

except gx.exceptions.DataContextError:
    checkpoint = context.checkpoints.add(
        gx.Checkpoint(
            name = "telco_checkpoint",
            validation_definitions = [validation_definition],
            actions=[action_list],
            result_format="SUMMARY",
        )
    )

In [16]:
# Execute the validation and get the results:

validation_results = checkpoint.run(
    batch_parameters = {"dataframe": df}
)

Calculating Metrics: 100%|██████████| 161/161 [00:00<00:00, 908.31it/s]


In [18]:
print(validation_results)

run_id={"run_name": null, "run_time": "2026-09-19T12:28:07.365214-04:00"} run_results={ValidationResultIdentifier::telco_expectations/__none__/20260919T162807.365214Z/pandas-Telco-Customer-Churn: {
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "pandas-Telco-Customer-Churn",
          "column": "customerID"
        },
        "meta": {},
        "id": "1e92b8c2-286a-494d-81a8-28610b100869",
        "severity": "critical"
      },
      "result": {
        "element_count": 7043,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
    

In [17]:
# Check the results and print a summary:

if validation_results.success:
    print("SUCCESS: All expectations passed! The dataframe is valid.")
else:
    print("FAILURE: Some expectations failed.")

SUCCESS: All expectations passed! The dataframe is valid.


In [21]:
# Get all unexpected rows after running Checkpoint:

# Retrieve the data validation Definition:
validation_definition = context.validation_definitions.get(definition_name)

In [24]:
for evr in validation_results.results:
    # Filter by status and type because get_unexpected_rows() supports only UnexpectedRowsExpectation
    if not evr.success and isinstance(evr.expectation, UnexpectedRowsExpectation):
        unexpected_rows = validation_definition.get_unexpected_rows(
            evr.expectation,
            batch_parameters=result.batch_parameters,
        )
        print(f"{len(unexpected_rows)} unexpected rows found")


AttributeError: 'CheckpointResult' object has no attribute 'results'